In [13]:
import numpy as np
import pandas as pd

In [14]:
df = pd.read_csv('../data/processed/cleaned_amazon_products.csv')
print(f"Loaded {len(df)} products.")

Loaded 8733 products.


In [15]:
minimum_ratings_threshold = 50 

popular_df = df[df['no_of_ratings'] >= minimum_ratings_threshold].copy()

popular_df['popularity_score'] = popular_df['ratings'] * popular_df['no_of_ratings']

popular_df = popular_df.sort_values(by='popularity_score', ascending=False)

# Let's look at the top 5!
display(popular_df[['name', 'ratings', 'no_of_ratings', 'popularity_score']].head())

,name,ratings,no_of_ratings,popularity_score
434,Amazon Basics High-Speed HDMI Cable - 10 Feet ...,4.4,437651.0,1925664.4
2329,"Amazon Basics High-Speed HDMI Cable, 6 Feet (2...",4.4,437651.0,1925664.4
1108,"Amazon Basics High-Speed HDMI Cable, 6 Feet - ...",4.4,437651.0,1925664.4
547,Amazon Basics Flexible Premium HDMI Cable (Bla...,4.4,437651.0,1925664.4
4506,AmazonBasics AAA Performance Alkaline Non-rech...,4.4,351441.0,1546340.4


In [16]:
def get_trending_products(dataframe, top_n=10):
    """
    Returns the top N trending products based on popularity score.
    """
    min_ratings = 50
    trending = dataframe[dataframe['no_of_ratings'] >= min_ratings].copy()
    
    trending['popularity_score'] = trending['ratings'] * trending['no_of_ratings']
    
    trending = trending.sort_values(by='popularity_score', ascending=False)
    
    return trending[['name', 'image', 'ratings', 'discount_price', 'link']].head(top_n)

#Let's get the Top 5 trending products
top_5_trending = get_trending_products(df, top_n=5)
display(top_5_trending)

,name,image,ratings,discount_price,link
434,Amazon Basics High-Speed HDMI Cable - 10 Feet ...,https://m.media-amazon.com/images/I/61GUctqz0-...,4.4,379.0,https://www.amazon.in/AmazonBasics-High-Speed-...
2329,"Amazon Basics High-Speed HDMI Cable, 6 Feet (2...",https://m.media-amazon.com/images/I/61ntykhzGV...,4.4,349.0,https://www.amazon.in/AmazonBasics-High-Speed-...
1108,"Amazon Basics High-Speed HDMI Cable, 6 Feet - ...",https://m.media-amazon.com/images/I/61pBvlYVPx...,4.4,269.0,https://www.amazon.in/AmazonBasics-High-Speed-...
547,Amazon Basics Flexible Premium HDMI Cable (Bla...,https://m.media-amazon.com/images/I/71gEiWJNPe...,4.4,269.0,https://www.amazon.in/AmazonBasics-Flexible-HD...
4506,AmazonBasics AAA Performance Alkaline Non-rech...,https://m.media-amazon.com/images/I/81F7OfBTCv...,4.4,639.0,https://www.amazon.in/AmazonBasics-Performance...


In [17]:
from IPython.display import HTML, display

def display_recommendations(rec_df):
    """Renders a nice HTML grid for our recommendations"""
    html = "<div style='display:flex; flex-wrap:wrap;'>"
    
    for idx, row in rec_df.iterrows():
        html += f"""
        <div style='margin: 10px; padding: 10px; border: 1px solid #ddd; border-radius: 8px; width: 200px; text-align: center;'>
            <img src='{row["image"]}' style='width: 150px; height: 150px; object-fit: contain;'>
            <h4 style='font-size: 14px; margin: 10px 0; height: 40px; overflow: hidden;'>{row["name"][:50]}...</h4>
            <p style='color: #E67E22; font-weight: bold;'>Rating: {row["ratings"]} ⭐</p>
            <p style='color: #27AE60; font-weight: bold;'>Price: ₹{row["discount_price"]}</p>
            <a href='{row["link"]}' target='_blank' style='background-color: #3498DB; color: white; padding: 5px 10px; text-decoration: none; border-radius: 4px;'>View Product</a>
        </div>
        """
    html += "</div>"
    display(HTML(html))

# Let's see the magic!
display_recommendations(top_5_trending)

# content based recommendation

In [18]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel


df = pd.read_csv('../data/processed/cleaned_amazon_products.csv')

df = df.reset_index(drop=True)

In [19]:
def clean_text(text):
    if not isinstance(text,str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]','',text)
    return text

df['tags'] = df['name'].apply(clean_text)

print("Original:", df['name'].iloc[0])
print("Cleaned: ", df['tags'].iloc[0])
    

Original: Redmi 10 Power (Power Black, 8GB RAM, 128GB Storage)
Cleaned:  redmi 10 power power black 8gb ram 128gb storage


In [20]:
df.shape

(8733, 8)

In [21]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['tags'])
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
print("Cosine Similarity Matrix Shape:", cosine_sim.shape)

TF-IDF Matrix Shape: (8733, 13014)
Cosine Similarity Matrix Shape: (8733, 8733)


In [22]:
indices = pd.Series(df.index, index=df['name']).drop_duplicates()

def get_content_recommendations(product_name, cosine_sim_matrix=cosine_sim, dataframe=df, top_n=5):
    """
    Takes a product name and returns the top N similar products.
    """
    if product_name not in indices:
        return "Product not found in database."
    idx = indices[product_name]
    if type(idx) is pd.Series:
        idx = idx.iloc[0]
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    product_indices = [i[0] for i in sim_scores]
    return dataframe.iloc[product_indices][['name', 'image', 'ratings', 'discount_price', 'link']]

In [23]:
# Copy an EXACT name from your dataframe. 
# For example, let's grab the name of the product at index 0
test_product_name = df['name'].iloc[3] 
print(f"Finding recommendations for: {test_product_name}\n")

# Get recommendations
similar_products = get_content_recommendations(test_product_name)

# Display them using the HTML function we built in Phase 2!
display_recommendations(similar_products)

Finding recommendations for: Samsung Galaxy M33 5G (Mystique Green, 6GB, 128GB Storage) | 6000mAh Battery | Upto 12GB RAM with RAM Plus | Travel Adapte...

